In [2]:
import os
import zipfile
from tqdm.notebook import tqdm
import cv2
import shutil
from pathlib import Path
import random


In [3]:
root_dir = r'Y:\ZHL\isds\PS'
data_list = [
    r'Y:\ZHL\isds\PS\task0725\ymt-2'
]


In [4]:
def zip_folder_to_path(source_folder, destination_zip):
    print(f"zip '{source_folder}' to '{destination_zip}' ... ")
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"finished!")

In [5]:
def yolo_add_risk(input_dir,output_dir):
    for file in tqdm(os.listdir(input_dir)):
        if file.endswith('.txt'):
            with open(os.path.join(input_dir, file), 'r') as f:
                lines = f.readlines()
            with open(os.path.join(output_dir, file), 'w') as f:
                for line in lines:
                    line = line.strip()
                    line = line[:1] + ' 4 0 0 0 0' + line[1:]

                    f.write(line + '\n')

In [6]:
def yolo_add_risk_dir(input_dir,output_dir):
    cam_name_list = [
        'cam_DA4930148', 
        # 'cam_DA5148680', 'cam_DA5148683', 'cam_DA5324645', 'cam_DA5324655', 'cam_DA6102933'
        ]
    for cam_name in cam_name_list:
        infer_dir = os.path.join(input_dir, cam_name+'_infer', 'labels')
        label_dir = os.path.join(output_dir, cam_name, 'labels')
        os.makedirs(label_dir, exist_ok=True)
        yolo_add_risk(infer_dir,label_dir)


        image_dir = os.path.join(input_dir, cam_name)
        image_zip_path = os.path.join(output_dir, cam_name+'_image.zip')
        label_zip_path = os.path.join(output_dir, cam_name+'_label.zip')
        zip_folder_to_path(
            source_folder=image_dir,
            destination_zip=image_zip_path
        )
        zip_folder_to_path(
            source_folder=label_dir,
            destination_zip=label_zip_path
        )

In [7]:
for data_path in data_list:
    input_dir = os.path.join(data_path, 'rectified_images', 'rectified_images')
    output_dir = os.path.join(data_path, 'track_data')
    yolo_add_risk_dir(input_dir,output_dir)

  0%|          | 0/10417 [00:00<?, ?it/s]

zip 'Y:\ZHL\isds\PS\task0725\ymt-2\rectified_images\rectified_images\cam_DA4930148' to 'Y:\ZHL\isds\PS\task0725\ymt-2\track_data\cam_DA4930148_image.zip' ... 
finished!
zip 'Y:\ZHL\isds\PS\task0725\ymt-2\track_data\cam_DA4930148\labels' to 'Y:\ZHL\isds\PS\task0725\ymt-2\track_data\cam_DA4930148_label.zip' ... 
finished!


In [9]:
def seg_to_bbox(points, img_w, img_h):
    """把 seg 多边形(0-1)转成 bbox 像素坐标"""
    xs = points[0::2]
    ys = points[1::2]
    xs = [int(x * img_w) for x in xs]
    ys = [int(y * img_h) for y in ys]
    return int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))


def crop_with_bbox(img, bbox, out_size=(256, 256)):
    """裁剪 bbox 并 resize"""
    h, w = img.shape[:2]
    x1, y1, x2, y2 = bbox
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w - 1, x2), min(h - 1, y2)
    if x2 <= x1 or y2 <= y1:
        return None
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return cv2.resize(crop, out_size)

def convert_yolo_txt_to_fastreid(frames_dir, labels_dir, out_dir="fastreid_dataset/signboard"):
    train_dir = os.path.join(out_dir, "bounding_box_train")
    query_dir = os.path.join(out_dir, "query")
    gallery_dir = os.path.join(out_dir, "bounding_box_test")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(query_dir, exist_ok=True)
    os.makedirs(gallery_dir, exist_ok=True)

    image_list = sorted(os.listdir(frames_dir))[:100]
    ids = []
    for frame_name in tqdm(image_list):
        label_name = Path(frame_name).with_suffix('.txt')

        frame_path = os.path.join(frames_dir, frame_name)
        if not os.path.exists(frame_path):
            continue

        img = cv2.imread(frame_path)
        if img is None:
            continue

        with open(os.path.join(labels_dir, label_name), "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            parts = line.strip().split(" ")

            class_id = int(parts[0])
            coords = list(map(float, parts[6:-1]))
            track_id = int(parts[-1])

            bbox = seg_to_bbox(coords, img.shape[1], img.shape[0])
            crop = crop_with_bbox(img, bbox)
            if crop is None:
                continue
            ids.append(track_id)

            save_name = f"{track_id:04d}_{frame_name.replace('.jpg','')}_{i}.jpg"
            save_path = os.path.join(train_dir, save_name)
            cv2.imwrite(save_path, crop)
    ids = list(set(ids))

    print(f"✅ 转换完成，{len(ids)} 训练集保存在 {train_dir}")
    print(f"⚠️ 下一步：拆分 query/gallery")

def split_query_gallery(train_dir, query_dir, gallery_dir):
    """每个ID抽一张图到query，其余到gallery"""
    id_to_imgs = {}
    for fname in os.listdir(train_dir):
        if not fname.endswith(".jpg"):
            continue
        track_id = fname.split("_")[0]
        id_to_imgs.setdefault(track_id, []).append(fname)

    for track_id, imgs in id_to_imgs.items():
        if len(imgs) == 0:
            continue
        query_img = random.choice(imgs)
        shutil.copy(os.path.join(train_dir, query_img), os.path.join(query_dir, query_img))
        for img in imgs:
            if img == query_img:
                continue
            shutil.copy(os.path.join(train_dir, img), os.path.join(gallery_dir, img))

    print(f"✅ 已拆分 query({len(os.listdir(query_dir))}) / gallery({len(os.listdir(gallery_dir))})")

In [10]:
for data_path in data_list:
    cam_name_list = [
        'cam_DA4930148', 
        # 'cam_DA5148680',
        # 'cam_DA5148683', 
        # 'cam_DA5324645', 
        # 'cam_DA5324655', 
        # 'cam_DA6102933'
        ]
    for cam_name in cam_name_list:
        images_dir = os.path.join(data_path, 'rectified_images', 'rectified_images', cam_name)
        labels_dir = os.path.join(data_path, 'track_data', cam_name, 'labels')
        output_dir = os.path.join(data_path, 'track_data', cam_name, 'reid')
        convert_yolo_txt_to_fastreid(images_dir, labels_dir, output_dir)
        split_query_gallery(
            train_dir=os.path.join(output_dir, "bounding_box_train"),
            query_dir=os.path.join(output_dir, "query"),
            gallery_dir=os.path.join(output_dir, "bounding_box_test"),
        )

  0%|          | 0/100 [00:00<?, ?it/s]

✅ 转换完成，7 训练集保存在 Y:\ZHL\isds\PS\task0725\ymt-2\track_data\cam_DA4930148\reid\bounding_box_train
⚠️ 下一步：拆分 query/gallery
✅ 已拆分 query(7) / gallery(347)
